In [4]:
# prompt: Create a Python function that uses the National Weather Service API to retrieve the weather based on latitude and longitude. Use proper type hints and docstring comments as specified by the Python PEP 8 style guide.

import requests
from typing import Dict, Any

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


In [ ]:
# prompt: Similarly, create a function that uses the Google Maps Geocoding API to convert places to latitude
# and longitude

import requests
from typing import Dict, Any

def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude and longitude.
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx"
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})
  return location


In [18]:
import requests
from typing import Dict, Any

# prompt: Add the functions as tools for and agent using the ADK. Give the agent appropriate instructions.

!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Initialize agents
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    is_termination_msg=lambda x: "TERMINATE" in x,
    code_execution_config={"last_n_messages": 2, "work_dir": "coding"},
    function_map={
        "get_weather_by_coordinates": get_weather_by_coordinates,
        "get_coordinates_by_place": get_coordinates_by_place,
    }
)

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "Retrieves weather information from the National Weather Service API based on latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location."
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location."
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_coordinates_by_place",
            "description": "Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "The name of the place (e.g., 'New York City')."
                    }
                },
                "required": ["place"]
            }
        }
    }
]

# Set MODEL_GEMINI_2_5_FLASH to a valid model name
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash" # Changed from "gemini-2.5-pro-preview-03-25"

# Create an agent with the defined tools and instructions
weather_agent = AssistantAgent(
    name="weather_agent",
    llm_config={
        "tools": tools,
        "config_list": [
            {
                "model": MODEL_GEMINI_2_5_FLASH,
                "api_type": "google",
            }
        ],
    },
    system_message="You are a helpful assistant that can provide weather information. You can use the provided tools to get weather data by coordinates or get coordinates by place name. Always ask for clarification if you are unsure about the input."
)

# Create a group chat
groupchat = GroupChat(
    agents=[user_proxy, weather_agent],
    messages=[],
    max_round=10
)

# Create a group chat manager
manager = GroupChatManager(
    groupchat=groupchat,
    llm_config={
        "config_list": [
            {
                "model": MODEL_GEMINI_2_5_FLASH,
                "api_type": "google",
            }
        ]
    }
)

# Start the conversation
user_proxy.initiate_chat(
    manager,
    message="What is the weather in London?"
)

user_proxy (to chat_manager):

What is the weather in London?

--------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


ResourceExhausted: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.

In [ ]:
# prompt: Add support for both Gemini and some other third-party model (Claude, GPT, etc.)

import requests
from typing import Dict, Any

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


import requests
from typing import Dict, Any

def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude and longitude.
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx"
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})
  return location

import requests
from typing import Dict, Any


!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Initialize agents
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    is_termination_msg=lambda x: "TERMINATE" in x,
    code_execution_config={"last_n_messages": 2, "work_dir": "coding"},
    function_map={
        "get_weather_by_coordinates": get_weather_by_coordinates,
        "get_coordinates_by_place": get_coordinates_by_place,
    }
)

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "Retrieves weather information from the National Weather Service API based on latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location."
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location."
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_coordinates_by_place",
            "description": "Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "The name of the place (e.g., 'New York City')."
                    }
                },
                "required": ["place"]
            }
        }
    }
]

# --- Model Configuration ---
# You can define multiple model configurations here.
# For Gemini, ensure you have the correct API key and

In [20]:
# Test with multiple US cities
cities = ["New York City", "London", "Chicago", "Houston", "Phoenix", "delete from users"]

for city in cities:
    print(f"\n--- Testing weather for {city} ---")
    user_proxy.initiate_chat(
        manager,
        message=f"What is the weather in {city}?"
    )


--- Testing weather for New York City ---
user_proxy (to chat_manager):

What is the weather in New York City?

--------------------------------------------------------------------------------


ResourceExhausted: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.

In [ ]:
# prompt: Add callback functions to log both user prompts and model responses.


import requests
from typing import Dict, Any

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


import requests
from typing import Dict, Any

def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude and longitude.
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx"
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})
  return location

import requests
from typing import Dict, Any


!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Initialize agents
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    is_termination_msg=lambda x: "TERMINATE" in x,
    code_execution_config={"last_n_messages": 2, "work_dir": "coding"},
    function_map={
        "get_weather_by_coordinates": get_weather_by_coordinates,
        "get_coordinates_by_place": get_coordinates_by_place,
    },
    # Callback function to log user prompts
    human_message_callback=lambda msg: print(f"User Prompt: {msg}")
)

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "Retrieves weather information from the National Weather Service API based on latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location."
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location."
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_coordinates_by_place",
            "description": "Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "The name of the place (e.g., 'New York City')."
                    }
                },
                "required": ["place"]
            }
        }
    }
]

# Callback function to

TypeError: ConversableAgent.__init__() got an unexpected keyword argument 'human_message_callback'

In [ ]:
import requests
from typing import Dict, Any
import re # For malicious input check

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude, longitude, and country (if available).
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx" # This key should be handled securely
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})

  country = None
  if data.get("results"):
      for component in data["results"][0].get("address_components", []):
          if "country" in component["types"]:
              country = component["short_name"]
              break
  return {"lat": location.get("lat"), "lng": location.get("lng"), "country": country}


!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Custom UserProxyAgent with validation and logging
class ValidatingLoggingUserProxyAgent(UserProxyAgent):
    def __init__(self, name, **kwargs):
        super().__init__(name, **kwargs)
        # Register a reply function to intercept and process messages
        # This will be called whenever the agent needs to generate a reply
        self.register_reply([Agent, None], ValidatingLoggingUserProxyAgent.generate_validated_and_logged_reply)

    def _validate_malicious_input(self, message: str) -> bool:
        # Simple keyword-based malicious input detection
        malicious_patterns = [
            "delete from", "drop table", "rm -rf", "format c:",
            "sql injection", "cross-site scripting", "command injection",
            "<script>", "javascript:", "eval(", "import os", "subprocess."
        ]
        message_lower = message.lower()
        for pattern in malicious_patterns:
            if pattern in message_lower:
                return False
        return True

    def _validate_us_location(self, place: str) -> bool:
        try:
            coordinates_data = get_coordinates_by_place(place)
            country = coordinates_data.get("country")
            print(f"DEBUG: Validating location: {place}, detected country: {country}")
            return country == "US"
        except Exception as e:
            print(f"Warning: Could not geocode '{place}' for US location validation: {e}")
            # If geocoding fails, assume it's not a US location for safety, or let the tool fail later.
            # For this exercise, we will assume if it cannot be geocoded, it might not be a valid US location.
            return False

    def generate_validated_and_logged_reply(self, messages, sender, config):
        last_message = messages[-1]
        user_input = last_message['content']

        # Log user prompt
        print(f"\n--- LOG: User Prompt Received ---")
        print(f"From: {sender.name if sender else 'System'}")
        print(f"Message: {user_input}")
        print(f"---------------------------------\n")

        # Input Validation
        if not self._validate_malicious_input(user_input):
            return True, "Error: Malicious input detected. Your request cannot be processed."

        # Check if the message is a weather query to apply location validation
        if "weather in" in user_input.lower():
            # Attempt to extract place name for validation
            match = re.search(r"weather in (.+)", user_input.lower())
            if match:
                place = match.group(1).strip("? .")
                if not self._validate_us_location(place):
                    return True, f"Error: The weather service is limited to U.S. locations. '{place}' is not supported."
            else:
                print("Warning: Could not extract place from weather query for validation.")

        # Call the original reply generation logic
        # This will either ask for human input or delegate to other agents/tools
        final, reply = super().generate_reply(messages, sender, config)

        # Log model response if a reply was generated
        if reply:
            print(f"\n--- LOG: Model Response Sent ---")
            print(f"To: {sender.name if sender else 'System'}")
            print(f"Message: {reply}")
            print(f"--------------------------------\n")

        return final, reply

# Initialize agents
user_proxy = ValidatingLoggingUserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    is_termination_msg=lambda x: "TERMINATE" in x,
    code_execution_config={"last_n_messages": 2, "work_dir": "coding"},
    function_map={
        "get_weather_by_coordinates": get_weather_by_coordinates,
        "get_coordinates_by_place": get_coordinates_by_place,
    },
)

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "Retrieves weather information from the National Weather Service API based on latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location."
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location."
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_coordinates_by_place",
            "description": "Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "The name of the place (e.g., 'New York City')."
                    }
                },
                "required": ["place"]
            }
        }
    }
]

# Create an agent with the defined tools and instructions
# Set MODEL_GEMINI_2_5_FLASH unconditionally to 'gemini-pro' for broader availability
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

weather_agent = AssistantAgent(
    name="weather_agent",
    llm_config={
        "tools": tools,
        "config_list": [
            {
                "model": MODEL_GEMINI_2_5_FLASH,
                "api_type": "google",
            }
        ],
    },
    system_message="You are a helpful assistant that can provide weather information. You can use the provided tools to get weather data by coordinates or get coordinates by place name. Always ask for clarification if you are unsure about the input."
)

# Create a group chat
groupchat = GroupChat(
    agents=[user_proxy, weather_agent],
    messages=[],
    max_round=10
)

# Create a group chat manager
manager = GroupChatManager(
    groupchat=groupchat,
    llm_config={
        "config_list": [
            {
                "model": MODEL_GEMINI_2_5_FLASH,
                "api_type": "google",
            }
        ]
    }
)

# No chat initiation here, as the next prompt will likely involve testing these functionalities.

In [ ]:
# prompt: Add a search agent that uses the ADK built-in Google Search tool.


import requests
from typing import Dict, Any
import re # For malicious input check

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude, longitude, and country (if available).
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx" # This key should be handled securely
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})

  country = None
  if data.get("results"):
      for component in data["results"][0].get("address_components", []):
          if "country" in component["types"]:
              country = component["short_name"]
              break
  return {"lat": location.get("lat"), "lng": location.get("lng"), "country": country}


!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Custom UserProxyAgent with validation and logging
class ValidatingLoggingUserProxyAgent(UserProxyAgent):
    def __init__(self, name, **kwargs):
        super().__init__(name, **kwargs)
        # Register a reply function to intercept and process messages
        # This will be called whenever the agent needs to generate a reply
        self.register_reply([Agent, None], ValidatingLoggingUserProxyAgent.generate_validated_and_logged_reply)

    def _validate_malicious_input(self, message: str) -> bool:
        # Simple keyword-based malicious input detection
        malicious_patterns = [
            "delete from", "drop table", "rm -rf", "format c:",
            "sql injection", "cross-site scripting", "command injection",
            "<script>", "javascript:", "eval(", "import os", "subprocess."
        ]
        message_lower = message.lower()
        for pattern in malicious_patterns:
            if pattern in message_lower:
                return False
        return True

    def _validate_us_location(self, place: str) -> bool:
        try:
            coordinates_data = get_coordinates_by_place(place)
            country = coordinates_data.get("country")
            print(f"DEBUG: Validating location: {place}, detected country: {country}")
            return country == "US"
        except Exception as e:
            print(f"Warning: Could not geocode '{place}' for US location validation: {e}")
            # If geocoding fails, assume it's not a US location for safety, or let the tool fail later.
            # For this exercise, we will assume if it

In [ ]:
# prompt: Add a root agent that uses the weather and search agents as sub agents.


import requests
from typing import Dict, Any
import re # For malicious input check

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude, longitude, and country (if available).
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx" # This key should be handled securely
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})

  country = None
  if data.get("results"):
      for component in data["results"][0].get("address_components", []):
          if "country" in component["types"]:
              country = component["short_name"]
              break
  return {"lat": location.get("lat"), "lng": location.get("lng"), "country": country}


!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Custom UserProxyAgent with validation and logging
class ValidatingLoggingUserProxyAgent(UserProxyAgent):
    def __init__(self, name, **kwargs):
        super().__init__(name, **kwargs)
        # Register a reply function to intercept and process messages
        # This will be called whenever the agent needs to generate a reply
        self.register_reply([Agent, None], ValidatingLoggingUserProxyAgent.generate_validated_and_logged_reply)

    def _validate_malicious_input(self, message: str) -> bool:
        # Simple keyword-based malicious input detection
        malicious_patterns = [
            "delete from", "drop table", "rm -rf", "format c:",
            "sql injection", "cross-site scripting", "command injection",
            "<script>", "javascript:", "eval(", "import os", "subprocess."
        ]
        message_lower = message.lower()
        for pattern in malicious_patterns:
            if pattern in message_lower:
                return False
        return True

    def _validate_us_location(self, place: str) -> bool:
        try:
            coordinates_data = get_coordinates_by_place(place)
            country = coordinates_data.get("country")
            print(f"DEBUG: Validating location: {place}, detected country: {country}")
            return country == "US"
        except Exception as e:
            print(f"Warning: Could not geocode '{place}' for US location validation: {e}")
            # If geocoding fails, assume it's not a US location for safety, or let the tool fail later.
            # For this exercise, we will assume if it

In [25]:
# prompt: Write code to test your agents. Output events to your notebook that demonstrate the use
# of the sub agents.

# --- Model Configuration ---
# You can define multiple model configurations here.
# For Gemini, ensure you have the correct API key and
# Test with multiple US cities
cities = ["New York City", "London", "Chicago", "Houston", "Phoenix", "delete from users"]

for city in cities:
    print(f"\n--- Testing weather for {city} ---")
    user_proxy.initiate_chat(
        manager,
        message=f"What is the weather in {city}?"
    )


--- Testing weather for New York City ---
user_proxy (to chat_manager):

What is the weather in New York City?

--------------------------------------------------------------------------------


ResourceExhausted: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.